In [4]:
ls

 --help
 -cwd.e102715733
 -cwd.o102715733
'15_Chriatian Hernandez.pdf'*
 2015_incidence_by_age.pdf
 2018-12-04_cooper_export_for_project_264697_filtered/
 22189_extraction.dta
 329/
 330/
 331/
 5858/
 5861/
 5864/
 5867/
 5873/
 5876/
 5879/
 9812_BR.csv
 9812_BR_updates.csv
 9814_CH.csv
'='
 Active_model_bundle_version_id_fromDoc/
 Admissions/
'Annual Review for 2019'/
 Bubble_Map_2022.pdf
 Bubble_Map_2022_without_studycounts.pdf
 Conversion_Files_CSV.py
'Copy of Bundle409_Other Gyne_ICDs_color_coded.xlsx'
 DSR_final.xlsx
 DSR_pdf_names.ipynb
 DSR_references_2023.xlsx
 DSR_test.xlsx
'Data landscape 1101'/
 Data_moving.py
 Dental_Figures_Summary.docx
'Dental_Table&Figures_2017'/
 Desktop_2022/
'Draft_May162022_Mat_Land_US_MMWRAbortionOnly_forChristian - Copy.xlsx'
 Draft_May162022_Mat_Land_US_MMWRAbortionOnly_forChristian.xlsx
'Draft_May172022_Mat_Land_US_MMWRAbortionOnly_forChristian - Copy.xlsx'
 Draft_May172022_Mat_Land_US_MMWRAbortionOnly_forChristian.xlsx
'ENDO_epi_lit_GBD2021_Oc

In [3]:
cd repos/live_birth_adjustments/

/mnt/share/homes/chrish47/repos/live_birth_adjustments


In [4]:
ls

README.md                python_shell.sh
dependency_map.csv       save.py
dependency_map_save.csv  slurm-62727550.out
epi_ids.csv              submit_jobs.py
get_best_model_status.R  upload_maternal_epi_bundles.py
maternal_core.py


In [36]:
############################################################################################################################
############################################################################################################################
############################################################################################################################

In [4]:
cd repos/live_birth_adjustments/

/mnt/share/homes/chrish47/repos/live_birth_adjustments


In [6]:
import subprocess
import pandas as pd
import numpy as np
import os
import re
from datetime import datetime
import time
from db_queries import get_best_model_versions
import getpass
import time
import gbd.constants as gbd

### TODO: Add a way to be able to submit Zero_Fistula and Fistula at the same
###       time, as currently the job would fail as the me_id for Fistula would
###       not have a model version id returned by get_model_versions, which is
###       used for the adjustments job name. Current workaround is to run
###       Zero_Fistula first (with zero=True) and once that all is done
###       then run Fistula (with zero=False).
################################################################################
# Please read the readme.txt in this repo. It will explain the whole strategy
################################################################################

# set round id, years to execute, and boolean for fistula
release_id = 16
yearvals = [1990, 1995, 2000, 2005, 2010, 2015, 2020, 2022, 2023]
zero = True #True#False

# set up stdout and stderr
username = getpass.getuser()
root = os.path.join('/share/scratch/users/', username)
error_path = "/mnt/share/scratch/users/chrish47/errors/%x.e%j"
output_path = "/mnt/share/scratch/users/chrish47/output/%x.o%j"

if not os.path.exists(error_path):
    os.makedirs(error_path)
if not os.path.exists(output_path):
    os.makedirs(output_path)

# pull in dependency map
dep_map = pd.read_csv('{}/dependency_map.csv'.format(os.getcwd()))

""" create new dataframe that converts all values in the dep_map to ints and 
matches each output_me with each of its input_mes """
columns = ['input_me','output_me']
extended_map = pd.DataFrame(columns=columns)
for index, row in dep_map.iterrows():
    ins = [int(x) for x in str(row.input_me).split(';')]
    outs = [int(x) for x in str(row.output_mes).split(';')]
    df = pd.DataFrame(columns=columns, data=list(zip(np.repeat(ins, len(outs)), 
        outs*len(ins)))) 
    extended_map = extended_map.append(df, ignore_index=True)
        
# make timestamped output folders
date_regex = re.compile('\W')
date_unformatted = str(datetime.now())[0:13]
c_date = date_regex.sub('_', date_unformatted)

#Automatic data folder vs Manual#
#base_dir = os.path.join(root, 'nonfatal_maternal', '{}'.format(c_date))
base_dir = os.path.join(root, 'nonfatal_maternal', '2024_01_10_14')
print(base_dir)

for output_me in extended_map.output_me.unique():
    out_dir = (os.path.join(base_dir, str(output_me)))
    if not os.path.exists(out_dir):
        os.makedirs(out_dir)

""" grab version ids for all input models. This strategy assumes that no new 
dismod source models have been marked best since the start of the LBA 
code. """
model_version_list = extended_map.input_me.unique()
mvid_df = get_best_model_versions(entity='modelable_entity',
    ids=model_version_list, release_id=release_id)


/tmp/ipykernel_1243430/325406172.py:51: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  extended_map = extended_map.append(df, ignore_index=True)
/tmp/ipykernel_1243430/325406172.py:51: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  extended_map = extended_map.append(df, ignore_index=True)
/tmp/ipykernel_1243430/325406172.py:51: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  extended_map = extended_map.append(df, ignore_index=True)
/tmp/ipykernel_1243430/325406172.py:51: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  extended_map = extended_map.append(df, ignore_index=True)
/tmp/ipykernel_1243430/325406172.py:51: FutureWarning: The frame.app

/share/scratch/users/chrish47/nonfatal_maternal/2024_01_10_14


In [41]:
################################################################################
# Zero out non-fistula locations and save results
################################################################################
if zero:
    class_name = "Zero_Fistula"
    input_me  = int(dep_map.loc[dep_map.class_name==class_name,
        'input_me'].item())
    output_me = int(dep_map.loc[dep_map.class_name==class_name,
        'output_mes'].item())

    # zero out locations
    job_list = []
    for year in yearvals:
        job_name = "{cn}_{y}".format(cn=class_name, y=year)
        job_list.append(job_name)
        call = ('sbatch -c 3 --mem=8G'
                ' -p long.q -C archive'
                ' -A proj_rgud'
                ' -t 12:30:00' 
                ' -o {o}'
                ' -e {e}'
                ' -J {jn}'
                ' python_shell.sh'
                ' maternal_core.py'
                ' "{arg1}" "{arg2}" "{arg3}" "{arg4}" "{arg5}" "{arg6}" '.format(
                    o=output_path, e=error_path, jn=job_name, 
                    arg1=class_name, arg2=base_dir, arg3=year, 
                    arg4=input_me, arg5=output_me, arg6=release_id))
        subprocess.check_output(call, shell=True)
        print(call)

sbatch -c 3 --mem=8G -p long.q -C archive -A proj_rgud -t 12:30:00 -o /mnt/share/scratch/users/chrish47/output/%x.o%j -e /mnt/share/scratch/users/chrish47/errors/%x.e%j -J Zero_Fistula_1990 python_shell.sh maternal_core.py "Zero_Fistula" "/share/scratch/users/chrish47/nonfatal_maternal/2024_01_10_14" "1990" "1552" "16535" "16" 
sbatch -c 3 --mem=8G -p long.q -C archive -A proj_rgud -t 12:30:00 -o /mnt/share/scratch/users/chrish47/output/%x.o%j -e /mnt/share/scratch/users/chrish47/errors/%x.e%j -J Zero_Fistula_1995 python_shell.sh maternal_core.py "Zero_Fistula" "/share/scratch/users/chrish47/nonfatal_maternal/2024_01_10_14" "1995" "1552" "16535" "16" 
sbatch -c 3 --mem=8G -p long.q -C archive -A proj_rgud -t 12:30:00 -o /mnt/share/scratch/users/chrish47/output/%x.o%j -e /mnt/share/scratch/users/chrish47/errors/%x.e%j -J Zero_Fistula_2000 python_shell.sh maternal_core.py "Zero_Fistula" "/share/scratch/users/chrish47/nonfatal_maternal/2024_01_10_14" "2000" "1552" "16535" "16" 
sbatch -c 

In [46]:
if zero:
    class_name = "Zero_Fistula"
    input_me  = int(dep_map.loc[dep_map.class_name==class_name,
        'input_me'].item())
    output_me = int(dep_map.loc[dep_map.class_name==class_name,
        'output_mes'].item())

    # zero out locations
    job_list = []

    hold = ",".join(job_list)
    model_version_ids = str(mvid_df.loc[mvid_df.modelable_entity_id==input_me, 
        'model_version_id'].item())
    save_job_name = "save_{meid}".format(meid=output_me)
    out_dir = ('{bd}/{meid}'.format(bd=base_dir, meid=output_me))
    call = ('sbatch -c 25 --mem=80G'
            ' -p long.q -C archive'
            ' -A proj_rgud'
            ' -t 12:30:00'
            ' -o {o}'
            ' -e {e}'
            ' -J {jn}'
            ' python_shell.sh'
            ' save.py'
            ' {arg1} \'{arg2}\' {arg3} {arg4}'.format(o=output_path, 
                e=error_path, jn=save_job_name, arg1=output_me, 
                arg2=model_version_ids, arg3=out_dir, arg4=release_id))
    subprocess.check_output(call, shell=True)
    print(call)
#print("test")

sbatch -c 25 --mem=80G -p long.q -C archive -A proj_rgud -t 12:30:00 -o /mnt/share/scratch/users/chrish47/output/%x.o%j -e /mnt/share/scratch/users/chrish47/errors/%x.e%j -J save_16535 python_shell.sh save.py 16535 '798053' /share/scratch/users/chrish47/nonfatal_maternal/2024_01_10_14/16535 16


In [17]:
################################################################################
# Adjust maternal causes for live births
################################################################################

# Run Adjustments
# remove the Zero_Fistula class or it will be uploaded twice
dep_map = dep_map.loc[dep_map.class_name != 'Zero_Fistula',:]
dep_map = dep_map.loc[dep_map.class_name != 'Zero_Fistula',:]
adjust_job_list = []
print("test1")
for index, row in dep_map.iterrows():
    class_name = row['class_name']
    input_me = row['input_me']
    output_me = row['output_mes']

    if class_name == "Fistula" and zero:
        hold = save_job_name
    else:
        hold = "no_holds"

    for year in yearvals:
        job_name = 'adj_{cn}_{y}'.format(cn=class_name, y=year)
        """ this line is to facilitate testing one class at a time, 
        change as needed """
        if (class_name not in ["abcdefghijk..."]):
            adjust_job_list.append(job_name)
            call = ('sbatch -c 4 --mem=5G'
                    ' -p long.q -C archive'
                    ' -A proj_rgud'
                    ' -t 12:30:00'
                    ' -o {o}'
                    ' -e {e}'
                    ' -J {jn}'
                    ' python_shell.sh'
                    ' maternal_core.py'
                    ' {arg1} {arg2} {arg3} "{arg4}" "{arg5}" {arg6}'.format(
                        o=output_path, e=error_path, jn=job_name, 
                        arg1=class_name, arg2=base_dir, arg3=year, 
                        arg4=input_me, arg5=output_me, arg6=release_id))
            subprocess.check_output(call, shell=True)
            print(call)
    print("test2")

test1
sbatch -c 4 --mem=5G -p long.q -C archive -A proj_rgud -t 12:30:00 -o /mnt/share/scratch/users/chrish47/output/%x.o%j -e /mnt/share/scratch/users/chrish47/errors/%x.e%j -J adj_Abortion_1990 python_shell.sh maternal_core.py Abortion /share/scratch/users/chrish47/nonfatal_maternal/2024_01_10_14 1990 "1555" "3644" 16
sbatch -c 4 --mem=5G -p long.q -C archive -A proj_rgud -t 12:30:00 -o /mnt/share/scratch/users/chrish47/output/%x.o%j -e /mnt/share/scratch/users/chrish47/errors/%x.e%j -J adj_Abortion_1995 python_shell.sh maternal_core.py Abortion /share/scratch/users/chrish47/nonfatal_maternal/2024_01_10_14 1995 "1555" "3644" 16
sbatch -c 4 --mem=5G -p long.q -C archive -A proj_rgud -t 12:30:00 -o /mnt/share/scratch/users/chrish47/output/%x.o%j -e /mnt/share/scratch/users/chrish47/errors/%x.e%j -J adj_Abortion_2000 python_shell.sh maternal_core.py Abortion /share/scratch/users/chrish47/nonfatal_maternal/2024_01_10_14 2000 "1555" "3644" 16
sbatch -c 4 --mem=5G -p long.q -C archive -A p

sbatch -c 4 --mem=5G -p long.q -C archive -A proj_rgud -t 12:30:00 -o /mnt/share/scratch/users/chrish47/output/%x.o%j -e /mnt/share/scratch/users/chrish47/errors/%x.e%j -J adj_Eclampsia_2023 python_shell.sh maternal_core.py Eclampsia /share/scratch/users/chrish47/nonfatal_maternal/2024_01_10_14 2023 "1544" "3635;2627" 16
test2
sbatch -c 4 --mem=5G -p long.q -C archive -A proj_rgud -t 12:30:00 -o /mnt/share/scratch/users/chrish47/output/%x.o%j -e /mnt/share/scratch/users/chrish47/errors/%x.e%j -J adj_Obstruct_1990 python_shell.sh maternal_core.py Obstruct /share/scratch/users/chrish47/nonfatal_maternal/2024_01_10_14 1990 "1550" "3641" 16
sbatch -c 4 --mem=5G -p long.q -C archive -A proj_rgud -t 12:30:00 -o /mnt/share/scratch/users/chrish47/output/%x.o%j -e /mnt/share/scratch/users/chrish47/errors/%x.e%j -J adj_Obstruct_1995 python_shell.sh maternal_core.py Obstruct /share/scratch/users/chrish47/nonfatal_maternal/2024_01_10_14 1995 "1550" "3641" 16
sbatch -c 4 --mem=5G -p long.q -C archi

NameError: name 'save_job_name' is not defined

In [18]:
# ############################################################################
# # Upload epi data
# ############################################################################

# hold = "no_holds"
####
dep_map = dep_map.loc[dep_map.class_name != 'Zero_Fistula',:]
dep_map = dep_map.loc[dep_map.class_name != 'Zero_Fistula',:]
adjust_job_list = []
print("test1")
for index, row in dep_map.iterrows():
    class_name = row['class_name']
    input_me = row['input_me']
    output_me = row['output_mes']

    if class_name == "Fistula" and zero:
        hold = 'skip'#save_job_name #Christian: This save_job_name variables is not needed, since we are skipping anyways.
    else:
        hold = "no_holds"
#####
    print("test3")
    hold = ",".join(adjust_job_list)
    dismod_dict = {
        2624 : ('maternal_sepsis', 377),
        2627 : ('maternal_htn', 827),
        1546 : ('maternal_htn', 829)
    }
    output_mes = [int(x) for x in str(output_me).split(';')]
    epi_upload_list = [x for x in output_mes if x in list(dismod_dict.keys())]
    
    for output_me in epi_upload_list:
        bundle_tuple = dismod_dict[output_me]
        """ this line is to facilitate testing one bundle at a time, 
        change to == as needed """
        if (int(bundle_tuple[1]) > -1):
            data_dir = os.path.join(base_dir, str(output_me))
            call = ('sbatch -c 3 --mem=10G'
                    ' -p long.q -C archive'
                    ' -A proj_rgud'
                    ' -t 12:30:00'
                    ' -o {o}'
                    ' -e {e}'
                    ' -J upload_bundle_{jn}'
                    ' python_shell.sh'
                    ' upload_maternal_epi_bundles.py'
                    ' {arg1} {arg2} "{arg3}" '.format(o=output_path,# Deleted decomp_step, no longer needed for elmo functions 
                        e=error_path, jn=bundle_tuple[1], arg1=data_dir, 
                        arg2=bundle_tuple[0], arg3=bundle_tuple[1]))
            subprocess.check_output(call, shell=True)
            print(call)
    print("test4")

test1
test3
test4
test3
test4
test3
sbatch -c 3 --mem=10G -p long.q -C archive -A proj_rgud -t 12:30:00 -o /mnt/share/scratch/users/chrish47/output/%x.o%j -e /mnt/share/scratch/users/chrish47/errors/%x.e%j -J upload_bundle_829 python_shell.sh upload_maternal_epi_bundles.py /share/scratch/users/chrish47/nonfatal_maternal/2024_01_10_14/1546 maternal_htn "829" 
test4
test3
sbatch -c 3 --mem=10G -p long.q -C archive -A proj_rgud -t 12:30:00 -o /mnt/share/scratch/users/chrish47/output/%x.o%j -e /mnt/share/scratch/users/chrish47/errors/%x.e%j -J upload_bundle_827 python_shell.sh upload_maternal_epi_bundles.py /share/scratch/users/chrish47/nonfatal_maternal/2024_01_10_14/2627 maternal_htn "827" 
test4
test3
test4
test3
test4
test3
sbatch -c 3 --mem=10G -p long.q -C archive -A proj_rgud -t 12:30:00 -o /mnt/share/scratch/users/chrish47/output/%x.o%j -e /mnt/share/scratch/users/chrish47/errors/%x.e%j -J upload_bundle_377 python_shell.sh upload_maternal_epi_bundles.py /share/scratch/users/chrish4

In [9]:
############################################################################
# Save results in parallel for everything except dismod_dict keys
############################################################################

    # hold = "no_holds"
    
####
dep_map = dep_map.loc[dep_map.class_name != 'Zero_Fistula',:]
dep_map = dep_map.loc[dep_map.class_name != 'Zero_Fistula',:]
adjust_job_list = []
print("test1")
for index, row in dep_map.iterrows():
    class_name = row['class_name']
    input_me = row['input_me']
    output_me = row['output_mes']

    if class_name == "Fistula" and zero:
        hold = 'skip'#save_job_name #Christian: This save_job_name variables is not needed, since we are skipping anyways.
    else:
        hold = "no_holds"
        
    hold = ",".join(adjust_job_list)
    dismod_dict = {
        2624 : ('maternal_sepsis', 377),
        2627 : ('maternal_htn', 827),
        1546 : ('maternal_htn', 829)
    }
    output_mes = [int(x) for x in str(output_me).split(';')]
    epi_upload_list = [x for x in output_mes if x in list(dismod_dict.keys())]
#####

    print("test5")
    hold = ",".join(adjust_job_list)
    save_mes = [x for x in output_mes if x not in list(dismod_dict.keys())]
    for me in save_mes:
        data_dir = os.path.join(base_dir, str(me))
        """ this line is to facilitate testing one bundle at a time, 
        change as needed """
        print(data_dir)
        if me not in [-1]:
            model_ids_str_for_save = ''
            df = extended_map.loc[extended_map.output_me==me,:]
            for index,row in df.iterrows():
                model_ids_str_for_save += ' meid {}, mvid {};'.format(
                    row.input_me, 
                    mvid_df.loc[mvid_df.modelable_entity_id==row.input_me,
                    'model_version_id'].item())
            model_ids_str_for_save = model_ids_str_for_save[:-1] + "."
            call = ('sbatch -c 25 --mem=80G'
                    ' -p long.q -C archive'
                    ' -A proj_rgud'
                    ' -t 12:30:00'
                    ' -o {o}'
                    ' -e {e}'
                    ' -J save_{jn}'
                    ' python_shell.sh'
                    ' save.py'
                    ' {arg1} \'{arg2}\' {arg3} "{arg4}" '.format(o=output_path, 
                        e=error_path, jn=me, arg1=me, 
                        arg2=model_ids_str_for_save, 
                        arg3=data_dir, arg4=release_id))
            subprocess.check_output(call, shell=True)
            print(call)
        print("test6")
    #check_output
    sleeptime = 5
    print("Sleeping for {} seconds".format(sleeptime))
    time.sleep(sleeptime)

test1
test5
/share/scratch/users/chrish47/nonfatal_maternal/2024_01_10_14/3644
sbatch -c 25 --mem=80G -p long.q -C archive -A proj_rgud -t 12:30:00 -o /mnt/share/scratch/users/chrish47/output/%x.o%j -e /mnt/share/scratch/users/chrish47/errors/%x.e%j -J save_3644 python_shell.sh save.py 3644 ' meid 1555, mvid 793241.' /share/scratch/users/chrish47/nonfatal_maternal/2024_01_10_14/3644 "16" 
test6
Sleeping for 5 seconds
test5
/share/scratch/users/chrish47/nonfatal_maternal/2024_01_10_14/3620
sbatch -c 25 --mem=80G -p long.q -C archive -A proj_rgud -t 12:30:00 -o /mnt/share/scratch/users/chrish47/output/%x.o%j -e /mnt/share/scratch/users/chrish47/errors/%x.e%j -J save_3620 python_shell.sh save.py 3620 ' meid 1535, mvid 798046.' /share/scratch/users/chrish47/nonfatal_maternal/2024_01_10_14/3620 "16" 
test6
/share/scratch/users/chrish47/nonfatal_maternal/2024_01_10_14/1536
sbatch -c 25 --mem=80G -p long.q -C archive -A proj_rgud -t 12:30:00 -o /mnt/share/scratch/users/chrish47/output/%x.o%j 